# Predicción de reclamaciones de seguros de viaje

Proyecto de **clasificación binaria** para predecir si una póliza tendrá una reclamación (`Claim`).

**Métrica principal:** F1, debido al desequilibrio de clases.

El análisis incluye regresión logística, balanceo mediante pesos de clase, upsampling, downsampling, árbol de decisión, ajuste del umbral y curva ROC/AUC-ROC.

> El dataset original no se incluye en GitHub. Coloca `travel_insurance_us.csv` en `data/`.


## 1. Carga y exploración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

paths = [
    Path("../data/travel_insurance_us.csv"),
    Path("data/travel_insurance_us.csv"),
    Path("/datasets/travel_insurance_us.csv")
]
path = next((p for p in paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("Coloca travel_insurance_us.csv en data/")

data = pd.read_csv(path)
print("Dimensiones:", data.shape)
display(data.head(10))


In [ ]:
display(data.dtypes.to_frame("tipo"))
display(data.isna().sum().sort_values(ascending=False).to_frame("faltantes"))
print("Duplicados:", data.duplicated().sum())
display(data["Claim"].value_counts().to_frame("cantidad"))
display(data["Claim"].value_counts(normalize=True).to_frame("proporción"))

data["Claim"].value_counts().plot(kind="bar")
plt.title("Distribución de reclamaciones")
plt.xlabel("Claim")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.show()


## 2. Regresión logística con One-Hot Encoding

Las variables categóricas se convierten mediante `pd.get_dummies(..., drop_first=True)`. La partición es 75/25 con `random_state=12345`.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

data_ohe = pd.get_dummies(data, drop_first=True)
target = data_ohe["Claim"]
features = data_ohe.drop(columns=["Claim"])

features_train, features_valid, target_train, target_valid = train_test_split(
    features, target, test_size=0.25, random_state=12345
)

model = LogisticRegression(solver="liblinear", random_state=12345)
model.fit(features_train, target_train)
predicted_valid = model.predict(features_valid)

print("Accuracy:", accuracy_score(target_valid, predicted_valid))
print("Precision:", precision_score(target_valid, predicted_valid, zero_division=0))
print("Recall:", recall_score(target_valid, predicted_valid, zero_division=0))
print("F1:", f1_score(target_valid, predicted_valid, zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(target_valid, predicted_valid)).plot()
plt.title("Matriz de confusión")
plt.show()


## 3. Modelo constante y desequilibrio

In [ ]:
constant_pred = pd.Series(0, index=target_valid.index)
print("Accuracy del modelo constante:",
      accuracy_score(target_valid, constant_pred))


## 4. Regresión logística con pesos balanceados

In [ ]:
balanced_model = LogisticRegression(
    solver="liblinear", random_state=12345, class_weight="balanced"
)
balanced_model.fit(features_train, target_train)
balanced_pred = balanced_model.predict(features_valid)

print("Accuracy:", accuracy_score(target_valid, balanced_pred))
print("Precision:", precision_score(target_valid, balanced_pred, zero_division=0))
print("Recall:", recall_score(target_valid, balanced_pred, zero_division=0))
print("F1:", f1_score(target_valid, balanced_pred, zero_division=0))


## 5. Upsampling

In [ ]:
from sklearn.utils import shuffle

def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    features_up = pd.concat([features_zeros] + [features_ones] * repeat)
    target_up = pd.concat([target_zeros] + [target_ones] * repeat)
    return shuffle(features_up, target_up, random_state=12345)

features_up, target_up = upsample(features_train, target_train, 10)
up_model = LogisticRegression(solver="liblinear", random_state=12345)
up_model.fit(features_up, target_up)
up_pred = up_model.predict(features_valid)

print("F1:", f1_score(target_valid, up_pred, zero_division=0))
print("Precision:", precision_score(target_valid, up_pred, zero_division=0))
print("Recall:", recall_score(target_valid, up_pred, zero_division=0))


## 6. Downsampling

In [ ]:
def downsample(features, target, fraction):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    features_down = pd.concat([
        features_zeros.sample(frac=fraction, random_state=12345),
        features_ones
    ])
    target_down = pd.concat([
        target_zeros.sample(frac=fraction, random_state=12345),
        target_ones
    ])
    return shuffle(features_down, target_down, random_state=12345)

features_down, target_down = downsample(features_train, target_train, 0.1)
down_model = LogisticRegression(solver="liblinear", random_state=12345)
down_model.fit(features_down, target_down)
down_pred = down_model.predict(features_valid)

print("F1:", f1_score(target_valid, down_pred, zero_division=0))
print("Precision:", precision_score(target_valid, down_pred, zero_division=0))
print("Recall:", recall_score(target_valid, down_pred, zero_division=0))


## 7. Ajuste del umbral

In [ ]:
probabilities = balanced_model.predict_proba(features_valid)[:, 1]

threshold_results = []
for threshold in np.arange(0, 0.31, 0.02):
    pred = probabilities > threshold
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(target_valid, pred, zero_division=0),
        "recall": recall_score(target_valid, pred, zero_division=0),
        "f1": f1_score(target_valid, pred, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_results)
display(threshold_results)

plt.figure(figsize=(9,5))
plt.plot(threshold_results["threshold"], threshold_results["precision"], label="Precision")
plt.plot(threshold_results["threshold"], threshold_results["recall"], label="Recall")
plt.plot(threshold_results["threshold"], threshold_results["f1"], label="F1")
plt.xlabel("Umbral")
plt.ylabel("Métrica")
plt.title("Métricas según el umbral")
plt.legend()
plt.show()


## 8. Curva ROC y AUC-ROC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(target_valid, probabilities)
auc = roc_auc_score(target_valid, probabilities)

plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--", label="Aleatorio")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC")
plt.legend()
plt.show()

print(f"AUC-ROC: {auc:.3f}")


## 9. Árbol de decisión con codificación ordinal

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

encoder = OrdinalEncoder()
ordinal = pd.DataFrame(encoder.fit_transform(data), columns=data.columns)

target_ord = ordinal["Claim"]
features_ord = ordinal.drop(columns=["Claim"])

X_train, X_valid, y_train, y_valid = train_test_split(
    features_ord, target_ord, test_size=0.25, random_state=12345
)

tree = DecisionTreeClassifier(random_state=12345)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, tree_pred))
print("Precision:", precision_score(y_valid, tree_pred, zero_division=0))
print("Recall:", recall_score(y_valid, tree_pred, zero_division=0))
print("F1:", f1_score(y_valid, tree_pred, zero_division=0))


## 10. Comparación final

In [ ]:
comparison = pd.DataFrame([
    ["Logística", accuracy_score(target_valid,predicted_valid), precision_score(target_valid,predicted_valid,zero_division=0), recall_score(target_valid,predicted_valid,zero_division=0), f1_score(target_valid,predicted_valid,zero_division=0)],
    ["Logística balanceada", accuracy_score(target_valid,balanced_pred), precision_score(target_valid,balanced_pred,zero_division=0), recall_score(target_valid,balanced_pred,zero_division=0), f1_score(target_valid,balanced_pred,zero_division=0)],
    ["Upsampling", accuracy_score(target_valid,up_pred), precision_score(target_valid,up_pred,zero_division=0), recall_score(target_valid,up_pred,zero_division=0), f1_score(target_valid,up_pred,zero_division=0)],
    ["Downsampling", accuracy_score(target_valid,down_pred), precision_score(target_valid,down_pred,zero_division=0), recall_score(target_valid,down_pred,zero_division=0), f1_score(target_valid,down_pred,zero_division=0)],
    ["Árbol de decisión", accuracy_score(y_valid,tree_pred), precision_score(y_valid,tree_pred,zero_division=0), recall_score(y_valid,tree_pred,zero_division=0), f1_score(y_valid,tree_pred,zero_division=0)]
], columns=["Modelo","Accuracy","Precision","Recall","F1"]).sort_values("F1", ascending=False)
display(comparison)
print(f"AUC-ROC: {auc:.3f}")


## Conclusiones

La selección del modelo debe priorizar F1 porque las clases están desbalanceadas. El notebook compara diferentes estrategias para mejorar la detección de reclamaciones y muestra el efecto de modificar el umbral de clasificación.

Los resultados concretos se generan al ejecutar el notebook con el dataset original.
